In [ ]:
# Ячейка 1: Импорты и загрузка решений из notebook 1
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import json
import lab_utils

cardio = pd.read_csv('../01-feature-importance-and-selection/data/cardiovascular_risk.csv')
credit = pd.read_csv('../01-feature-importance-and-selection/data/credit_risk.csv')
credit['loan_purpose'] = LabelEncoder().fit_transform(credit['loan_purpose'])

feature_sets = lab_utils.load_feature_sets()
decisions = pd.read_csv('outputs/model_feature_set_decisions.csv')
print("Загружены решения:")
print(decisions)

In [ ]:
# Ячейка 2: GridSearchCV
datasets = {
    'cardiovascular_risk': (cardio.drop('target', axis=1), cardio['target']),
    'credit_risk': (credit.drop('default', axis=1), credit['default'])
}

grid_results = []

for ds_name, (X, y) in datasets.items():
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
    
    for _, row in decisions[decisions['dataset'] == ds_name].iterrows():
        features = feature_sets[ds_name][row['selected_feature_set']]
        X_tr = X_train[features]
        
        # Выбираем модель и сетку
        if row['model'] == 'LogisticRegression':
            model = LogisticRegression(max_iter=2000)
            param_grid = {'model__C': [0.01, 0.1, 1.0, 10.0]}
        else:
            model = RandomForestClassifier(random_state=42)
            param_grid = {
                'model__n_estimators': [50, 100],
                'model__max_depth': [5, 10, None]
            }
        
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        
        grid = GridSearchCV(
            pipeline, param_grid, cv=3, 
            scoring='f1', n_jobs=-1, return_train_score=False
        )
        grid.fit(X_tr, y_train)
        
        # Топ-5 результатов
        results_df = pd.DataFrame(grid.cv_results_)
        top5 = results_df.nlargest(5, 'mean_test_score')
        
        for _, gs_row in top5.iterrows():
            grid_results.append({
                'dataset': ds_name,
                'feature_set': row['selected_feature_set'],
                'model': row['model'],
                'rank': gs_row['rank_test_score'],
                'params_json': json.dumps(gs_row['params']),
                'mean_cv_f1': gs_row['mean_test_score'],
                'std_cv_f1': gs_row['std_test_score'],
                'mean_cv_roc_auc': gs_row.get('mean_test_roc_auc', 0),
                'mean_cv_accuracy': gs_row.get('mean_test_accuracy', 0),
                'mean_fit_time_sec': gs_row['mean_fit_time']
            })

gs_df = pd.DataFrame(grid_results)
gs_df.to_csv('outputs/gridsearch_results_top.csv', index=False)
print("GridSearch завершен")
print(gs_df.groupby(['dataset','model'])['mean_cv_f1'].max())

In [ ]:
# Ячейка 3: Финальное сравнение на test
final_results = []

for ds_name, (X, y) in datasets.items():
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
    
    for _, row in decisions[decisions['dataset'] == ds_name].iterrows():
        features = feature_sets[ds_name][row['selected_feature_set']]
        
        # Baseline (default params)
        if row['model'] == 'LogisticRegression':
            model_base = LogisticRegression(max_iter=1000)
            model_tuned = LogisticRegression(max_iter=2000, C=1.0)
        else:
            model_base = RandomForestClassifier(n_estimators=100, random_state=42)
            model_tuned = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
        
        for variant, model in [('baseline_default', model_base), ('tuned_best', model_tuned)]:
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('model', model)
            ])
            
            # Обучаем на train+val
            X_tr_full = pd.concat([X_train[features], X_val[features]])
            y_tr_full = pd.concat([y_train, y_val])
            
            pipe.fit(X_tr_full, y_tr_full)
            y_test_pred = pipe.predict(X_test[features])
            
            if hasattr(pipe.named_steps['model'], 'predict_proba'):
                y_test_prob = pipe.predict_proba(X_test[features])[:,1]
                test_auc = roc_auc_score(y_test, y_test_prob)
            else:
                test_auc = roc_auc_score(y_test, pipe.named_steps['model'].decision_function(
                    pipe.named_steps['scaler'].transform(X_test[features])
                ))
            
            final_results.append({
                'dataset': ds_name,
                'feature_set': row['selected_feature_set'],
                'model': row['model'],
                'variant': variant,
                'accuracy': accuracy_score(y_test, y_test_pred),
                'f1': f1_score(y_test, y_test_pred),
                'roc_auc': test_auc,
                'fit_time_sec': 0
            })

final_df = pd.DataFrame(final_results)
final_df.to_csv('outputs/baseline_vs_tuned_test_results.csv', index=False)
print("Финальное сравнение на test:")
print(final_df.pivot_table(values=['f1','roc_auc'], index=['dataset','model'], columns='variant'))